In [7]:
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit import Qubit, Clbit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.result import marginal_counts
from qiskit.visualization import plot_histogram

In [2]:
# Magic numbers
OPT_LEVEL = 0
SHOTS = 4096
SEED = 7

BASIS_1Q = ['h','x','z','s','sdg','ry','rz','id']
BASIS_2Q = ['cx']
BASIS = BASIS_1Q + BASIS_2Q
LAM1, LAM2 = 0.005, 0.02
P = 0.03

In [3]:
# Helper functions
def make_noise_model(kind):
    """kind in {'ideal','gates','readout','combined'} -> NoiseModel"""
    nm = NoiseModel()
    if kind in ('gates', 'combined'):
        nm.add_all_qubit_quantum_error(depolarizing_error(LAM1, 1), BASIS_1Q)
        nm.add_all_qubit_quantum_error(depolarizing_error(LAM2, 2), BASIS_2Q)
    if kind in ('readout', 'combined'):
        nm.add_all_qubit_readout_error(ReadoutError([[1 - P, P],[P, 1 - P]]))
    return nm

def run_counts(qc, kind, shots=4096, seed=7):
    """QuantumCircuit, noise kind -> (counts dict, transpiled circuit)"""
    sim = AerSimulator(method="density_matrix",
                       noise_model=make_noise_model(kind),
                       basis_gates=BASIS)
    tqc = transpile(qc, sim, optimization_level=0, seed_transpiler=seed)
    result = sim.run(tqc, shots=shots, seed_simulator=seed).result()
    return result.get_counts(), tqc, result   # keep result for marginal_counts

def success_prob(counts, good_keys):
    return sum(counts.get(k, 0) for k in good_keys) / sum(counts.values())

In [ ]:
def quantum_teleportation(probe):
    # q[0] = qubit to be sent, q[1..2] = e-bits, q[3..5] = clbit
    q = QuantumCircuit(3, 3)
    q.initialize(probe, 0)

    # create entangled pair between q[1] and q[2]
    q.h(1)
    q.cx(1, 2)

    # entangle "unknown" probe state
    q.cx(0, 1)

    # measure on alice side
    q.h(0)
    q.measure(0, 0)
    q.measure(1, 1)

    # bob corrects
    with q.if_test((0, 1)):
        q.x(2)
    with q.if_test((0, 1)):
        q.z(2)
    q.measure(2, 2)
    print(q.draw())

In [12]:
quantum_teleportation([1/np.sqrt(2), 1/np.sqrt(2)])

     ┌─────────────────────────────┐          ┌───┐┌─┐                         »
q_0: ┤ Initialize(0.70711,0.70711) ├───────■──┤ H ├┤M├─────────────────────────»
     └────────────┬───┬────────────┘     ┌─┴─┐└┬─┬┘└╥┘                         »
q_1: ─────────────┤ H ├───────────────■──┤ X ├─┤M├──╫──────────────────────────»
                  └───┘             ┌─┴─┐└───┘ └╥┘  ║   ┌──────  ┌───┐ ───────┐»
q_2: ───────────────────────────────┤ X ├───────╫───╫───┤ If-0  ─┤ X ├  End-0 ├»
                                    └───┘       ║   ║   └──╥───  └───┘ ───────┘»
                                                ║   ║ ┌────╨────┐              »
c: 3/═══════════════════════════════════════════╩═══╩═╡ c_0=0x1 ╞══════════════»
                                                1   0 └─────────┘              »
«                                   
«q_0: ──────────────────────────────
«                                   
«q_1: ──────────────────────────────
«        ┌──────  ┌───┐ ───────┐ ┌─┐
«q_2: